In [2]:
import pandas as pd
import numpy as np

In [3]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

print(nav.shape)
print(transactions.shape)
print(performance.shape)

(46000, 3)
(32778, 13)
(40, 19)


In [4]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

print(nav.shape)
print(transactions.shape)
print(performance.shape)

(46000, 3)
(32778, 13)
(40, 19)


In [5]:
print(nav.columns)

print(transactions.columns)

print(performance.columns)

Index(['amfi_code', 'date', 'nav'], dtype='object')
Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='object')
Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='object')


In [7]:
# Convert date column
nav['date'] = pd.to_datetime(nav['date'])

# Sort values
nav = nav.sort_values(['amfi_code', 'date'])

# Remove duplicates
nav = nav.drop_duplicates()

# Forward fill missing NAV values
nav['nav'] = nav.groupby('amfi_code')['nav'].ffill()

# Remove invalid NAV values
nav = nav[nav['nav'] > 0]

print(nav.info())
print(nav.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 46000 entries, 5750 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.4 MB
None
amfi_code    0
date         0
nav          0
dtype: int64


In [8]:
nav.to_csv("../data/processed/clean_nav.csv", index=False)

print("clean_nav.csv saved successfully")

clean_nav.csv saved successfully


In [9]:
transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date']
)

transactions['transaction_type'] = (
    transactions['transaction_type']
    .str.upper()
    .str.strip()
)

transactions = transactions[
    transactions['amount_inr'] > 0
]

transactions = transactions.drop_duplicates()

print(transactions['transaction_type'].unique())
print(transactions['kyc_status'].unique())

['SIP' 'REDEMPTION' 'LUMPSUM']
['Verified' 'Pending']


In [10]:
performance = performance.drop_duplicates()

performance['expense_ratio_pct'] = pd.to_numeric(
    performance['expense_ratio_pct'],
    errors='coerce'
)

performance['sharpe_ratio'] = pd.to_numeric(
    performance['sharpe_ratio'],
    errors='coerce'
)

negative_sharpe = performance[
    performance['sharpe_ratio'] < 0
]

invalid_expense = performance[
    (performance['expense_ratio_pct'] < 0.1) |
    (performance['expense_ratio_pct'] > 2.5)
]

print("Negative Sharpe:", len(negative_sharpe))
print("Invalid Expense Ratio:", len(invalid_expense))

Negative Sharpe: 0
Invalid Expense Ratio: 0


In [11]:
transactions.to_csv(
    "../data/processed/clean_transactions.csv",
    index=False
)

performance.to_csv(
    "../data/processed/clean_performance.csv",
    index=False
)

print("Files saved successfully!")

Files saved successfully!


In [12]:
from sqlalchemy import create_engine

In [13]:
engine = create_engine("sqlite:///../bluestock_mf.db")

print("Database engine created successfully!")

Database engine created successfully!


In [14]:
nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

print("fact_nav loaded successfully!")

fact_nav loaded successfully!


In [15]:
transactions.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

print("fact_transactions loaded successfully!")

fact_transactions loaded successfully!


In [16]:
performance.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

print("fact_performance loaded successfully!")

fact_performance loaded successfully!
